# ZS601 会议室 3DGS 几何质量提升 —— 阶段 0/1/2 全流程（Colab）


**首次使用/收到更新通知时**：只需运行第 0 段「自更新」单元格，自动从 GitHub 拉取并打开最新 notebook；之后按顺序运行其余单元格。

In [ ]:
# 0) 自更新：从 GitHub v2-dev 拉取最新 notebook 并自动加载（旧版/zip 版 notebook 运行一次即可换新）
# 若自动跳转失败，运行结束后点击输出里的链接手动打开最新版
import urllib.request, os, json
os.makedirs("/content/gs_work", exist_ok=True)
NB_NEW = "/content/gs_work/ZS601_3DGS_stagesv2_latest.ipynb"
url = "https://raw.githubusercontent.com/VISjudy/ZS601_3DGS/v2-dev/gaussian-splattingWithMask_v2/colab/ZS601_3DGS_stagesv2.ipynb"
try:
    urllib.request.urlretrieve(url, NB_NEW)
    cur = os.path.abspath(__file__) if "__file__" in dir() else None
    print("[ok] 最新 notebook 已下载到:", NB_NEW)
    try:
        from google.colab import _message
        _message.blocking_request("navigate", {"url": "/notebooks" + NB_NEW}, timeout_sec=10)
        print("[ok] 已跳转到最新版 notebook，请从第 1 段开始运行")
    except Exception:
        print("[提示] 自动跳转失败：点击左侧文件面板打开 ZS601_3DGS_stagesv2_latest.ipynb，或用上方路径手动打开")
except Exception as e:
    print("[err] 下载失败:", e)

In [ ]:
# ============ 配置（按需修改） ============
# 数据 zip 在 Drive 中的路径（我的云端硬盘/LCCDataset/ZS601meetingroom/ZS601meetingroom_data.zip）
DATA_ZIP_IN_DRIVE = "/content/drive/MyDrive/LCCDataset/ZS601meetingroom/ZS601meetingroom_data.zip"
# 代码仓库（GitHub 公开仓库，v2-dev 分支）：更新代码只需我 push 后重跑「拉取代码+解压数据」单元格，无需改这里
REPO_URL = "https://github.com/VISjudy/ZS601_3DGS.git"
# 结果回传到 Drive 的目录
RESULTS_IN_DRIVE = "/content/drive/MyDrive/LCCDataset/ZS601meetingroom/results"

# Colab 本地工作目录（临时盘，读写快）
WORK = "/content/gs_work"
CODE_DIR = f"{WORK}/repo/gaussian-splattingWithMask_v2"   # git clone 后代码位置
DATA_DIR = f"{WORK}/dataset"   # 解压后由下方单元格自动校正
import os; os.makedirs(WORK, exist_ok=True)
print("配置完成")

In [ ]:
# 1) 挂载 Google Drive（会弹出授权链接，按提示完成）
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash -s "$DATA_ZIP_IN_DRIVE" "$WORK"
# 2) 复制数据 zip 到 Colab 临时盘（代码改从 GitHub 仓库拉取，见下一单元格）
set -e
cp "$1" "$2/data.zip"
ls -lh "$2"

In [ ]:
# 3) 拉取代码（GitHub）+ 解压数据，并自动定位数据根目录（含 sparse/0 的那一层）
# 只拉 v2-dev 分支（v2 开发分支）；更新代码：我 push 后重跑本单元格即可（先删旧 clone 再拉最新）
import zipfile, os, glob, shutil
# 先切到根目录：若 shell 停留在旧 repo 内（上一轮安装单元格 %cd 进去的），
# 直接删除会把当前目录删掉，导致 git clone 报 getcwd 错误
os.chdir("/")
if os.path.exists(f"{WORK}/repo"):
    shutil.rmtree(f"{WORK}/repo")
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
!git clone --depth 1 -b v2-dev "$REPO_URL" "$WORK/repo"
assert os.path.isdir(f"{WORK}/repo/gaussian-splattingWithMask_v2"), "clone 结果缺少 gaussian-splattingWithMask_v2 目录，请检查仓库结构"

with zipfile.ZipFile(f"{WORK}/data.zip") as zf:
    zf.extractall(f"{WORK}/data_raw")

# 自动寻找包含 sparse/images.txt 的目录，其上一级即数据根目录
cands = glob.glob(f"{WORK}/data_raw/**/sparse/images.txt", recursive=True)
assert cands, "解压结果中找不到 sparse/images.txt，请检查 zip 结构"
DATA_DIR = os.path.dirname(os.path.dirname(cands[0]))

# 创建 sparse/0/ 目录并把三个 .txt 文件拷贝进去（保留 .bin 原文件不动）
sparse0 = os.path.join(DATA_DIR, "sparse", "0")
os.makedirs(sparse0, exist_ok=True)
for name in ["cameras.txt", "images.txt", "points3D.txt"]:
    src = os.path.join(DATA_DIR, "sparse", name)
    assert os.path.exists(src), f"sparse/ 下缺少 {name}，请检查数据包"
    shutil.copy2(src, os.path.join(sparse0, name))

print("CODE_DIR =", CODE_DIR)
print("DATA_DIR =", DATA_DIR)
print("sparse/0/ 内容:", sorted(os.listdir(sparse0)))

In [ ]:
# 4) 安装依赖：轻量库 + 编译 CUDA 子模块（光栅化器编译约 5~10 分钟，属正常现象）
# 阶段2：光栅化器已改造（新增 normal 法向图输出通道），务必用 v2 代码重新编译；
# 先卸载旧版再安装（避免混用 v1 旧光栅化器 / 断线重连后的残留），未安装过会自动跳过
!pip uninstall -y diff-gaussian-rasterization 2>/dev/null || true
!pip install laspy matplotlib plyfile --quiet
!cd "$CODE_DIR" && pip install ./submodules/simple-knn --quiet
!cd "$CODE_DIR" && pip install ./submodules/diff-gaussian-rasterization

# fused-ssim 用 clone 方式安装（仓库未随代码 zip 分发）；失败也不影响训练，会自动回退普通 ssim
%cd "$CODE_DIR/submodules"
!git clone https://github.com/rahul-goel/fused-ssim.git || echo "clone 失败（可选组件，自动回退普通 ssim）"
%cd fused-ssim
!pip install . --no-build-isolation || echo "fused-ssim 安装失败（可选组件，自动回退普通 ssim）"
%cd "$CODE_DIR"

# 安装验证：必须通过，否则后续训练会立即报 ModuleNotFoundError。
# 编译失败常见原因：1) 运行时类型不是 GPU（「代码执行程序 -> 更改运行时类型」选 T4）
# 2) 本单元格被跳过或中途打断——重新完整运行本单元格即可
try:
    from diff_gaussian_rasterization import GaussianRasterizer
    from simple_knn._C import distCUDA2
    print("[ok] diff-gaussian-rasterization 与 simple-knn 安装成功")
except ImportError as e:
    raise RuntimeError("CUDA 子模块安装未完成：{}\n请上翻查看编译报错，修复后重新运行本单元格".format(e))

In [ ]:
# 4.1) 编译诊断（安装单元格报错时运行）：确认 GPU/CUDA 环境 + 直接编译光栅化器，打印日志最后 60 行
# pip 的报错常被截断，真正的 nvcc 错误要看 setup.py 完整日志；诊断通过后回到上一单元格重新 pip install
!nvidia-smi -L; nvcc --version | tail -1
!cd "$CODE_DIR/submodules/diff-gaussian-rasterization" && rm -rf build *.egg-info && python setup.py bdist_wheel > /tmp/build.log 2>&1; tail -n 60 /tmp/build.log

## 阶段 0：数据预处理（一次性）
产物：替换后的 `points3D.txt/ply`（含颜色+法向量）、`images.txt/images_test.txt/images-val10.txt`、10 张法向量方向图（`intermediate/normal_visual/`）

In [ ]:
!cd "$CODE_DIR" && python preprocess.py --data_path "$DATA_DIR" --seed 42 --val_num 10 --test_ratio 0.05
# 验收：把 10 张法向量方向图与 GT 拼图展示
import glob
from IPython.display import display
from PIL import Image
for p in sorted(glob.glob(f"{DATA_DIR}/intermediate/normal_visual/val*_normal.png")):
    gt = p.replace("_normal.png", "_gt.png")
    a, b = Image.open(p).resize((240, 415)), Image.open(gt).resize((240, 415))
    canvas = Image.new("RGB", (485, 415))
    canvas.paste(a, (0, 0)); canvas.paste(b, (245, 0))
    display(canvas)

## 阶段 1-C① 冒烟验证（2000 轮，跑通全链路）
验收点：初始化打印各轴 scale 统计（z 轴极小）、`intermediate/init_2d/` 导出 ply、`val_render/iter_1/` 起每 500 轮渲染、`loss_log.csv` 与曲线图、`evaluate.py` 出指标。

注：`--data_device cpu` 把训练图像放 CPU 内存而非显存；`--lazy_load --lazy_cache 500` 让图片按需从磁盘读取、最多缓存 500 张，防止 2694 张图全部常驻内存导致爆 RAM（数据已放在 Colab 本地盘，读取开销很小）。

In [ ]:
!cd "$CODE_DIR" && python train_mask.py \
  -s "$DATA_DIR" -m output/smoke_init2d \
  --init_2d --iterations 2000 \
  --val_file "$DATA_DIR/sparse/0/images-val10.txt" \
  --test_iterations 2000 --save_iterations 2000 \
  --alpha_masks masks --data_device cpu --lazy_load --lazy_cache 500 --disable_viewer

In [ ]:
# 冒烟产物检查（渲染图/法向图/GT 仅首轮、loss 记录、val 指标记录、init_2d 导出）
!ls -R "$CODE_DIR/output/smoke_init2d/val_render" | head -40
!head -3 "$CODE_DIR/output/smoke_init2d/loss_log.csv"
!cat "$CODE_DIR/output/smoke_init2d/val_metrics.csv"
!ls "$DATA_DIR/intermediate/init_2d"

from IPython.display import display
from PIL import Image
import glob, os
display(Image.open(f"{CODE_DIR}/output/smoke_init2d/loss_curves_2000.png"))
# 最后一轮 val 图对比（每组从左到右：render | gt | normal，GT 只在首轮保存）
last_dir = sorted(glob.glob(f"{CODE_DIR}/output/smoke_init2d/val_render/iter_*"),
                  key=lambda x: int(x.split('_')[-1]))[-1]
for i in range(10):
    rp = f"{last_dir}/val{i}_render.png"
    gp = f"{last_dir}/val{i}_gt.png"
    nmp = f"{last_dir}/val{i}_normal.png"
    if not os.path.exists(rp):
        continue
    paths = [q for q in [rp, gp, nmp] if os.path.exists(q)]
    imgs = [Image.open(q).resize((220, 380)) for q in paths]
    canvas = Image.new("RGB", (225 * len(imgs) - 5, 380))
    for k, im in enumerate(imgs):
        canvas.paste(im, (225 * k, 0))
    display(canvas)


In [ ]:
# 冒烟评估（图像指标 + cloud2cloud 几何指标）
!cd "$CODE_DIR" && python evaluate.py -m output/smoke_init2d -s "$DATA_DIR"

## 阶段 1-C② 正式训练（150000 轮，三组对比）
**三组训练超参完全一致，变量只有两个开关 `--init_2d`（2D 椭球初始化）与 `--freeze_2d_z`(训练中形状约束，z 轴冻结)**：
- A 组 baseline：标准 3DGS 初始化（两个开关都不开）
- B 组 init2d+freeze：2D 椭球初始化 + 训练中保持扁平（法向对齐 + z 轴冻结）
- C 组 init2d_free：仅 2D 椭球初始化，不约束形状（椭球厚度可自由优化）

⚠️ 每组需数小时。Colab 免费版可能中途断连，建议开 Pro 或用「断点续训」（`--start_checkpoint`）。
训练中途可查看 `output/*/val_render/` 的渲染图与 `val_metrics.csv` 的指标/点数/耗时变化。


In [ ]:
# A 组：baseline（标准初始化）
!cd "$CODE_DIR" && python train_mask.py \
  -s "$DATA_DIR" -m output/zs601_baseline \
  --iterations 150000 --sh_degree 2 \
  --position_lr_init 0.000016 --position_lr_final 0.00000016 --position_lr_max_steps 150000 \
  --scaling_lr 0.0015 \
  --densification_interval 10000 --densify_until_iter 100000 \
  --opacity_reset_interval 150000 --densify_grad_threshold 0.0002 \
  --seed 42 --alpha_masks masks --data_device cpu --lazy_load --lazy_cache 500 --disable_viewer \
  --val_file "$DATA_DIR/sparse/0/images-val10.txt" \
  --test_iterations 50000 100000 150000 --save_iterations 50000 100000 150000

In [ ]:
# B 组：init_2d + freeze_2d_z（2D 椭球初始化 + 训练中形状约束保持扁平），其余超参与 A 组完全一致
!cd "$CODE_DIR" && python train_mask.py \
  -s "$DATA_DIR" -m output/zs601_init2d \
  --init_2d --freeze_2d_z \
  --iterations 150000 --sh_degree 2 \
  --position_lr_init 0.000016 --position_lr_final 0.00000016 --position_lr_max_steps 150000 \
  --scaling_lr 0.0015 \
  --densification_interval 10000 --densify_until_iter 100000 \
  --opacity_reset_interval 150000 --densify_grad_threshold 0.0002 \
  --seed 42 --alpha_masks masks --data_device cpu --lazy_load --lazy_cache 500 --disable_viewer \
  --val_file "$DATA_DIR/sparse/0/images-val10.txt" \
  --test_iterations 50000 100000 150000 --save_iterations 50000 100000 150000


In [ ]:
# C 组：仅 init_2d（2D 椭球初始化，不加形状约束，椭球厚度可自由优化），其余超参与 A 组完全一致
!cd "$CODE_DIR" && python train_mask.py \
  -s "$DATA_DIR" -m output/zs601_init2d_free \
  --init_2d \
  --iterations 150000 --sh_degree 2 \
  --position_lr_init 0.000016 --position_lr_final 0.00000016 --position_lr_max_steps 150000 \
  --scaling_lr 0.0015 \
  --densification_interval 10000 --densify_until_iter 100000 \
  --opacity_reset_interval 150000 --densify_grad_threshold 0.0002 \
  --seed 42 --alpha_masks masks --data_device cpu --lazy_load --lazy_cache 500 --disable_viewer \
  --val_file "$DATA_DIR/sparse/0/images-val10.txt" \
  --test_iterations 50000 100000 150000 --save_iterations 50000 100000 150000


In [ ]:
# 三组分别评估并打印对比表
!cd "$CODE_DIR" && python evaluate.py -m output/zs601_baseline -s "$DATA_DIR"
!cd "$CODE_DIR" && python evaluate.py -m output/zs601_init2d -s "$DATA_DIR"
!cd "$CODE_DIR" && python evaluate.py -m output/zs601_init2d_free -s "$DATA_DIR"

import json
rows = []
for name in ["zs601_baseline", "zs601_init2d", "zs601_init2d_free"]:
    try:
        with open(f"{CODE_DIR}/output/{name}/metrics.json") as f:
            m = json.load(f)
    except FileNotFoundError:
        print(f"[提示] {name} 尚未训练完成，跳过")
        continue
    im, g = m["image_metrics"], m["geometry"]
    rows.append([name, f"{im['PSNR']:.3f}", f"{im['L1']:.5f}", f"{im['SSIM']:.4f}",
                 f"{g['mean']:.4f}", f"{g['median']:.4f}", f"{g['rmse']:.4f}", f"{g['p90']:.4f}"])
hdr = ["组别", "PSNR", "L1", "SSIM", "geo_mean", "geo_median", "geo_rmse", "geo_p90"]
print("\n" + " | ".join(f"{h:>16}" for h in hdr))
for r in rows:
    print(" | ".join(f"{c:>16}" for c in r))


In [ ]:
# 结果回传 Drive（排除体积大的 point_cloud ply，保留指标/曲线/渲染图；需要 ply 可自行去掉 --exclude）
!mkdir -p "$RESULTS_IN_DRIVE"
!rsync -a --exclude "point_cloud" "$CODE_DIR/output" "$RESULTS_IN_DRIVE/"
!rsync -a "$DATA_DIR/intermediate" "$RESULTS_IN_DRIVE/"
print("结果已回传：", RESULTS_IN_DRIVE)

## 阶段 2：法向量约束 loss（2DGS 思路）
- **2-A**：光栅化器新增 normal 输出通道（每高斯法向 = 旋转矩阵第 3 列，朝向相机翻转，alpha 加权合成），反向梯度回传四元数
- **2-B**：`--lambda_normal`（渲染法向 vs 深度导出法向一致性，1-|cos| 均值，mask 外像素不参与）+ `--lambda_scale`（z 轴厚度正则 |exp(scaling)[:,2]| 均值）
- **2-C**：val 每 100 轮输出 RGB + 法向方向图（(n+1)/2），与阶段 0 点云法向投影图（intermediate/normal_visual/）可对照
- **2-D**：`--lambda_size`（形状约束防大椭球：逐高斯最长尺度轴 exp 后均值的正则，默认 0 关闭，建议 0.01 起调）；与 2-B 互补——2-B 只压法向轴厚度，此项限制切向铺展
- **2-E（高斯级形态正则，目标：小扁盘贴表面 + 邻居法向趋同）**：`--lambda_flat`（每高斯最薄轴尺度正则，强制每个高斯至少一轴薄，不依赖轴序号）+ `--lambda_smooth`（kNN 邻居法向趋同 mean(1-|dot|)，`--smooth_knn_k`=8 / `--smooth_every`=500 可调）；均默认 0，建议冒烟 `--lambda_flat 0.05 --lambda_smooth 0.05`
- loss_log.csv 新增 `normal_loss`、`scale_reg`、`size_reg`、`flat_reg`、`smooth_reg` 五列（共 10 列）；每 10 轮 print 一次各正则项

⚠️ 注意：本阶段起 `--lambda_scale/--lambda_normal` 默认 0.1/0.1，若需重跑 A/B 组 baseline 请显式加 `--lambda_scale 0 --lambda_normal 0`；
法向 loss 对齐 2DGS **延迟启用**（`--normal_start_iter`，2DGS 默认 7000；本项目激光点云初始化几何已准，正式训练传 1000、冒烟传 0）；
深度靶标为 **median depth**（光栅化器新通道，累计 alpha 首次越过 0.5 处的高斯深度，比期望逆深度锐利，对齐 2DGS）；`--lambda_size` 默认 0，仅需要防大椭球时显式开启


In [ ]:
# 阶段2冒烟（2000 轮）：init_2d + 法向约束 loss；确认 normal 图输出、Normal Loss/Scale Reg 打印、csv 新增两列
# --normal_start_iter 0：冒烟专用（正式训练默认 7000 轮后才启用法向 loss，对齐 2DGS）
!cd "$CODE_DIR" && python train_mask.py \
  -s "$DATA_DIR" -m output/zs601_smoke_normal \
  --init_2d --lambda_scale 0.1 --lambda_normal 0.1 --normal_start_iter 0 \
  --iterations 2000 --sh_degree 2 \
  --position_lr_init 0.000016 --position_lr_final 0.00000016 --position_lr_max_steps 2000 \
  --scaling_lr 0.0015 \
  --densification_interval 10000 --densify_until_iter 100000 \
  --opacity_reset_interval 150000 --densify_grad_threshold 0.0002 \
  --seed 42 --alpha_masks masks --data_device cpu --lazy_load --lazy_cache 500 --disable_viewer \
  --val_file "$DATA_DIR/sparse/0/images-val10.txt" \
  --test_iterations 2000 --save_iterations 2000


In [ ]:
# 阶段2冒烟产物检查：loss_log.csv 7 列表头、val_render 法向图、normal loss 数值
!head -3 "$CODE_DIR/output/zs601_smoke_normal/loss_log.csv"
!tail -5 "$CODE_DIR/output/zs601_smoke_normal/loss_log.csv"
!ls "$CODE_DIR/output/zs601_smoke_normal/val_render" | head
import pandas as pd, os
df = pd.read_csv(f"{CODE_DIR}/output/zs601_smoke_normal/loss_log.csv")
assert "normal_loss" in df.columns and "scale_reg" in df.columns, "loss_log.csv 缺少新增列！"
print("normal_loss: 首行 %.6f -> 末行 %.6f" % (df.normal_loss.iloc[0], df.normal_loss.iloc[-1]))
print("scale_reg:   首行 %.6f -> 末行 %.6f" % (df.scale_reg.iloc[0], df.scale_reg.iloc[-1]))
from PIL import Image
from IPython.display import display
d = f"{CODE_DIR}/output/zs601_smoke_normal/val_render"
iters = sorted([x for x in os.listdir(d) if x.startswith("iter_")], key=lambda s: int(s.split("_")[1]))
last = os.path.join(d, iters[-1])
imgs = [Image.open(os.path.join(last, f"val0_{k}.png")).resize((320, 554)) for k in ("render", "normal")]
canvas = Image.new("RGB", (655, 560), (40, 40, 40))
canvas.paste(imgs[0], (5, 3)); canvas.paste(imgs[1], (330, 3))
display(canvas)  # 左=RGB 渲染，右=法向方向图（RGB=(n+1)/2，+x红/+y绿/+z蓝）


In [ ]:
# 阶段2正式训练：C 组（init_2d + 法向约束 loss），配置同阶段1，输出 output/expC/
!cd "$CODE_DIR" && python train_mask.py \
  -s "$DATA_DIR" -m output/expC \
  --init_2d --lambda_scale 0.1 --lambda_normal 0.1 --normal_start_iter 1000 \
  --iterations 150000 --sh_degree 2 \
  --position_lr_init 0.000016 --position_lr_final 0.00000016 --position_lr_max_steps 150000 \
  --scaling_lr 0.0015 \
  --densification_interval 10000 --densify_until_iter 100000 \
  --opacity_reset_interval 150000 --densify_grad_threshold 0.0002 \
  --seed 42 --alpha_masks masks --data_device cpu --lazy_load --lazy_cache 500 --disable_viewer \
  --val_file "$DATA_DIR/sparse/0/images-val10.txt" \
  --test_iterations 50000 100000 150000 --save_iterations 50000 100000 150000 \
  --checkpoint_iterations 50000 100000 150000


In [ ]:
# 阶段2 D 组正式训练：baseline 标准初始化 + 法向约束 loss（不传 --init_2d / --freeze_2d_z）
# --lambda_size 为形状约束（防大椭球：逐高斯惩罚最长尺度轴均值），建议 0.01 起调，改 0 可关闭；
# 其余超参与 A/B/C 组完全一致。chkpnt 保存在 output/zs601_expD/，断线恢复方式同 C 组
!cd "$CODE_DIR" && python train_mask.py \
  -s "$DATA_DIR" -m output/zs601_expD \
  --lambda_scale 0.1 --lambda_normal 0.1 --lambda_size 0.01 --normal_start_iter 1000 \
  --iterations 150000 --sh_degree 2 \
  --position_lr_init 0.000016 --position_lr_final 0.00000016 --position_lr_max_steps 150000 \
  --scaling_lr 0.0015 \
  --densification_interval 10000 --densify_until_iter 100000 \
  --opacity_reset_interval 150000 --densify_grad_threshold 0.0002 \
  --seed 42 --alpha_masks masks --data_device cpu --lazy_load --lazy_cache 500 --disable_viewer \
  --val_file "$DATA_DIR/sparse/0/images-val10.txt" \
  --test_iterations 50000 100000 150000 --save_iterations 50000 100000 150000 \
  --checkpoint_iterations 50000 100000 150000

# D 组评估（图像指标 + cloud2cloud 几何指标）
!cd "$CODE_DIR" && python evaluate.py -m output/zs601_expD -s "$DATA_DIR"

### 断点续训（Colab 断开/超时后从这里恢复）
前提：训练命令带了 `--checkpoint_iterations`（chkpnt50000/100000/150000.pth 保存在 output/expC/）。
`/content` 是临时盘，断开即清空——所以训练中要定期把 chkpnt 同步到 Drive（下方第 1 个代码块），
重连后从 Drive 拉回（第 2 个代码块），再用 `--start_checkpoint` 从断点继续（第 3 个代码块）。
`restore` 会完整恢复模型参数+优化器状态+致密化统计+迭代轮数，从断点下一轮继续。


In [ ]:
# 训练中/训练后：把断点与日志同步到 Drive（chkpnt 单个约几百 MB，务必及时同步，/content 断开即清空）
!mkdir -p "$RESULTS_IN_DRIVE/expC_checkpoints"
!cp -u "$CODE_DIR/output/expC"/chkpnt*.pth "$RESULTS_IN_DRIVE/expC_checkpoints/" 2>/dev/null || echo "暂无 chkpnt 文件"
!cp -u "$CODE_DIR/output/expC/loss_log.csv" "$CODE_DIR/output/expC/val_metrics.csv" "$RESULTS_IN_DRIVE/expC_checkpoints/" 2>/dev/null || true
!ls -lh "$RESULTS_IN_DRIVE/expC_checkpoints"


In [ ]:
# 重连后（先跑过 挂载Drive/拉取代码/解压数据/编译 单元格）：从 Drive 拉回断点
import os
os.makedirs(f"{CODE_DIR}/output/expC", exist_ok=True)
!cp "$RESULTS_IN_DRIVE/expC_checkpoints"/chkpnt*.pth "$CODE_DIR/output/expC/"
!ls -lh "$CODE_DIR/output/expC/"


In [ ]:
# 从断点继续训练：其余参数必须与首次训练完全一致，仅多一行 --start_checkpoint
# 自动选用 Drive 中编号最大的 chkpnt（如 100000，则从第 100001 轮继续）
import glob, os
ckpts = sorted(glob.glob(f"{CODE_DIR}/output/expC/chkpnt*.pth"),
               key=lambda p: int(os.path.basename(p).replace("chkpnt", "").replace(".pth", "")))
assert ckpts, "没有找到 chkpnt 文件，请先运行上方拉回单元格"
CKPT = ckpts[-1]
print("从断点继续:", CKPT)

!cd "$CODE_DIR" && python train_mask.py \
  -s "$DATA_DIR" -m output/expC \
  --start_checkpoint "$CKPT" \
  --init_2d --lambda_scale 0.1 --lambda_normal 0.1 --normal_start_iter 1000 \
  --iterations 150000 --sh_degree 2 \
  --position_lr_init 0.000016 --position_lr_final 0.00000016 --position_lr_max_steps 150000 \
  --scaling_lr 0.0015 \
  --densification_interval 10000 --densify_until_iter 100000 \
  --opacity_reset_interval 150000 --densify_grad_threshold 0.0002 \
  --seed 42 --alpha_masks masks --data_device cpu --lazy_load --lazy_cache 500 --disable_viewer \
  --val_file "$DATA_DIR/sparse/0/images-val10.txt" \
  --test_iterations 50000 100000 150000 --save_iterations 50000 100000 150000 \
  --checkpoint_iterations 50000 100000 150000


In [ ]:
# C 组评估（图像指标 + cloud2cloud 几何指标）
!cd "$CODE_DIR" && python evaluate.py -m output/expC -s "$DATA_DIR"


In [ ]:
# 四组汇总：intermediate/ablation/summary.md（A/B/C/D 指标对比 + scale z 统计 + 同视角拼图）
# A/B 用阶段1产物（zs601_baseline / zs601_init2d），C 用阶段2产物（expC）
import json, os
import numpy as np
import torch
from PIL import Image

groups = {"A_baseline": "output/zs601_baseline", "B_init2d_freeze": "output/zs601_init2d", "C_init2d_normal": "output/expC", "D_baseline_normal": "output/zs601_expD"}
ab_dir = f"{DATA_DIR}/intermediate/ablation" if os.path.isdir(f"{DATA_DIR}/intermediate") else f"{CODE_DIR}/output/ablation"
os.makedirs(ab_dir, exist_ok=True)

lines = ["# A/B/C/D 四组消融对比（阶段1 + 阶段2）", ""]
lines.append("几何指标 = 高斯中心点云 vs 激光参考点云 cloud2cloud 距离（evaluate.py，已剔除非有限坐标）；"
             "图像指标 = test 集渲染 vs GT（mask 口径一致）。")
lines.append("")
lines.append("| 组别 | PSNR | L1 | SSIM | geo_mean | geo_median | geo_rmse | geo_p90 | scale_z mean | scale_z p90 |")
lines.append("|---|---|---|---|---|---|---|---|---|---|")
for name, mdir in groups.items():
    mpath = f"{CODE_DIR}/{mdir}/metrics.json"
    ply_path = f"{CODE_DIR}/{mdir}/point_cloud/iteration_150000/point_cloud.ply"
    if not os.path.exists(mpath):
        lines.append(f"| {name} | 未训练完成 | - | - | - | - | - | - | - | - |")
        continue
    m = json.load(open(mpath))
    im, g = m["image_metrics"], m["geometry"]
    # scale z 统计：读取最终 ply 的 scale_2（log 空间）并 exp
    sz_mean = sz_p90 = float("nan")
    if os.path.exists(ply_path):
        from plyfile import PlyData
        vd = PlyData.read(ply_path)["vertex"].data
        sz = np.exp(vd["scale_2"])
        sz_mean, sz_p90 = float(sz.mean()), float(np.percentile(sz, 90))
    lines.append("| {} | {:.3f} | {:.5f} | {:.4f} | {:.4f} | {:.4f} | {:.4f} | {:.4f} | {:.3e} | {:.3e} |".format(
        name, im["PSNR"], im["L1"], im["SSIM"], g["mean"], g["median"], g["rmse"], g["p90"], sz_mean, sz_p90))
lines.append("")
lines.append("## 结论（人工填写）")
lines.append("- C 相对 B 的有效性：几何指标 / 图像指标变化：____")
lines.append("- D 相对 A 的有效性（法向约束 loss）：几何指标 / 图像指标变化：____")
lines.append("--lambda_size 形状约束的效果（高斯数 / scale 分布 / 大椭球情况）：____")
open(f"{ab_dir}/summary.md", "w", encoding="utf-8").write("\n".join(lines))
print("summary.md ->", f"{ab_dir}/summary.md")
print(open(f"{ab_dir}/summary.md", encoding="utf-8").read())

# 同视角 RGB / 法向三组并排拼图（取各自 val_render 最后一轮的 val0）
def last_val_img(mdir, kind):
    d = f"{CODE_DIR}/{mdir}/val_render"
    if not os.path.isdir(d):
        return None
    iters = sorted([x for x in os.listdir(d) if x.startswith("iter_")], key=lambda s: int(s.split("_")[1]))
    fp = os.path.join(d, iters[-1], f"val0_{kind}.png")
    return Image.open(fp).resize((280, 484)) if os.path.exists(fp) else None

for kind in ("render", "normal"):
    imgs = [last_val_img(m, kind) for m in groups.values()]
    if any(i is None for i in imgs):
        print(f"[提示] {kind} 拼图跳过：有组缺少 val_render 产物")
        continue
    canvas = Image.new("RGB", (3 * 285 + 5, 490), (40, 40, 40))
    for k, im in enumerate(imgs):
        canvas.paste(im, (5 + k * 285, 3))
    out = f"{ab_dir}/compare_{kind}.png"
    canvas.save(out)
    print("拼图 ->", out)
    display(canvas)
